In [18]:
import pandas as pd
import numpy as np
import re
import gdown
import zipfile
import os
from pathlib import Path

# Gathering Data

In [2]:
def download_and_unzip_gdrive(file_id, output_dir="downloaded_files"):
    """
    Download a zip file from Google Drive and extract it
    
    Args:
        file_id: Google Drive file ID (from sharing link)
        output_dir: Directory to extract the files to
    """

    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Download the file
    zip_path = os.path.join(output_dir, "downloaded_file.zip")
    print(f"Downloading file with ID: {file_id}")
    print(f"This may take a while for a 100-200MB file...")
    
    try:
        # Download using gdown
        gdown.download(id=file_id, output=zip_path, quiet=False)
        
        # Check if download was successful
        if os.path.getsize(zip_path) == 0:
            raise Exception("Downloaded file is empty")
            
        print(f"Download complete! File size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
        
        # Unzip the file
        print("Extracting files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(output_dir)
            file_list = zip_ref.namelist()
            
        print(f"Extraction complete! Extracted {len(file_list)} files to '{output_dir}'")
        
        # Optional: Remove the zip file after extraction
        os.remove(zip_path)
        
        
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
file_id = '1xMGWU8I51cN-0Ntn3CUpAJkfeZPPn4if'
path = 'raw_data'

download_and_unzip_gdrive(file_id, path)

This may take a while for a 100-200MB file...


Downloading...
From (original): https://drive.google.com/uc?id=1xMGWU8I51cN-0Ntn3CUpAJkfeZPPn4if
From (redirected): https://drive.google.com/uc?id=1xMGWU8I51cN-0Ntn3CUpAJkfeZPPn4if&confirm=t&uuid=e8b9c168-26c5-4e76-baa8-ea46cd2d6d9f
To: /home/bonbon/code/data-science-capstone-github/test_data/downloaded_file.zip
100%|██████████| 161M/161M [07:51<00:00, 341kB/s] 


Download complete! File size: 153.29 MB
Extracting files...
Extraction complete! Extracted 4 files to 'test_data'


In [3]:
first_df = pd.read_csv('raw_data/postings.csv') # https://www.kaggle.com/datasets/arshkon/linkedin-job-postings
second_df = pd.read_csv('raw_data/linkedin_job_data.csv') # https://www.kaggle.com/datasets/shashankshukla123123/linkedin-job-data
third_df = pd.read_csv('raw_data/dataset_jotpars.csv', encoding='latin-1') # https://data.mendeley.com/datasets/8b62nm2gmv/1/files/5b29a121-c1a4-47eb-b31e-748bdc4b767f

# Assesing Data

In [4]:
display(first_df.info())
print("\n")

display(second_df.info())
print("\n")

display(third_df.info())
print("\n")

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  str    
 2   title                       123849 non-null  str    
 3   description                 123842 non-null  str    
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   str    
 6   location                    123849 non-null  str    
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  str    
 12  applies                     23320 non-null   float64
 13  original_listed_time     

None



<class 'pandas.DataFrame'>
RangeIndex: 7927 entries, 0 to 7926
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_ID              7927 non-null   int64
 1   job                 7894 non-null   str  
 2   location            7894 non-null   str  
 3   company_id          3 non-null      str  
 4   company_name        7889 non-null   str  
 5   work_type           7736 non-null   str  
 6   full_time_remote    7848 non-null   str  
 7   no_of_employ        7603 non-null   str  
 8   no_of_application   7887 non-null   str  
 9   posted_day_ago      7920 non-null   str  
 10  alumni              4861 non-null   str  
 11  Hiring_person       5717 non-null   str  
 12  linkedin_followers  4815 non-null   str  
 13  hiring_person_link  5719 non-null   str  
 14  job_details         7881 non-null   str  
 15  Column1             18 non-null     str  
 16  Unnamed: 16         2 non-null      str  
 17  Unna

None



<class 'pandas.DataFrame'>
RangeIndex: 8797 entries, 0 to 8796
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          8797 non-null   str    
 1   requirment    8797 non-null   str    
 2   description   8797 non-null   str    
 3   experience    8797 non-null   float64
 4   salary_range  8797 non-null   str    
dtypes: float64(1), str(4)
memory usage: 343.8 KB


None

In [8]:
display(first_df.duplicated(subset=['job_id']).sum())

display(second_df.duplicated(subset=['job_ID']).sum())

display(third_df.duplicated().sum())

np.int64(0)

np.int64(2084)

np.int64(1305)

In [11]:
duplicate_job = len(set(first_df['job_id']) & set(second_df['job_ID']))

display(duplicate_job)

0

# Cleaning Data

In [19]:
display(first_df['remote_allowed'].value_counts())

display(second_df['work_type'].value_counts())

display([third_df['salary_range'].head()])

remote_allowed
1.0    15246
Name: count, dtype: int64

work_type
Remote    2999
Name: count, dtype: int64

[0    106k-124k
 1      57k-60k
 2    107k-118k
 3      68k-70k
 4      46k-48k
 Name: salary_range, dtype: str]

In [20]:
def extract_salary(range_str):
    if pd.isna(range_str) or not isinstance(range_str, str):
        return pd.Series([np.nan, np.nan, np.nan])

    match = re.search(r'([\d\.]+)\s*([a-zA-Z]+)?\s*-\s*([\d\.]+)\s*([a-zA-Z]+)?', range_str)

    if not match:
        return pd.Series([np.nan, np.nan, np.nan])

    min_val, min_suffix, max_val, max_suffix = match.groups()

    def convert_to_number(num_str, suffix):
        try:
            num = float(num_str)
        except ValueError:
            return np.nan

        if suffix:
            suffix = suffix.lower()
            if suffix == 'k':
                num *= 1000
            elif suffix in ['lac', 'lacs', 'lakh']:
                num *= 100000
        return num

    min_salary = convert_to_number(min_val, min_suffix)
    max_salary = convert_to_number(max_val, max_suffix)

    med_salary = (min_salary + max_salary) / 2

    return pd.Series([med_salary])

In [21]:
first_df = first_df[first_df['remote_allowed'] == 1]

second_df = second_df[second_df['work_type'] == 'Remote']

third_df['med_salary'] = third_df['salary_range'].apply(extract_salary)

In [23]:
second_df.drop_duplicates(subset=['job_ID'], inplace=True)

third_df.drop_duplicates(inplace=True)

In [24]:
display(first_df.info())
print("\n")

display(second_df.info())
print("\n")

display(third_df.info())
print("\n")

<class 'pandas.DataFrame'>
Index: 15246 entries, 6 to 123847
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      15246 non-null  int64  
 1   company_name                14814 non-null  str    
 2   title                       15246 non-null  str    
 3   description                 15243 non-null  str    
 4   max_salary                  4318 non-null   float64
 5   pay_period                  4864 non-null   str    
 6   location                    15246 non-null  str    
 7   company_id                  14814 non-null  float64
 8   views                       14898 non-null  float64
 9   med_salary                  546 non-null    float64
 10  min_salary                  4318 non-null   float64
 11  formatted_work_type         15246 non-null  str    
 12  applies                     6415 non-null   float64
 13  original_listed_time        15246 non-null  fl

None



<class 'pandas.DataFrame'>
Index: 2371 entries, 0 to 7925
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_ID              2371 non-null   int64
 1   job                 2371 non-null   str  
 2   location            2371 non-null   str  
 3   company_id          0 non-null      str  
 4   company_name        2371 non-null   str  
 5   work_type           2371 non-null   str  
 6   full_time_remote    2356 non-null   str  
 7   no_of_employ        2329 non-null   str  
 8   no_of_application   2369 non-null   str  
 9   posted_day_ago      2369 non-null   str  
 10  alumni              1906 non-null   str  
 11  Hiring_person       1481 non-null   str  
 12  linkedin_followers  1246 non-null   str  
 13  hiring_person_link  1481 non-null   str  
 14  job_details         2364 non-null   str  
 15  Column1             1 non-null      str  
 16  Unnamed: 16         0 non-null      str  
 17  Unnamed: 

None



<class 'pandas.DataFrame'>
Index: 7492 entries, 0 to 8796
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          7492 non-null   str    
 1   requirment    7492 non-null   str    
 2   description   7492 non-null   str    
 3   experience    7492 non-null   float64
 4   salary_range  7492 non-null   str    
 5   med_salary    7492 non-null   float64
dtypes: float64(2), str(4)
memory usage: 409.7 KB


None

In [25]:
os.makedirs('analysis_data', exist_ok=True)

first_df.to_csv('analysis_data/first_data.csv')
second_df.to_csv('analysis_data/second_data.csv')
third_df.to_csv('analysis_data/third_data.csv')